Veneziano Milestone 4

Connecting to an API/Pulling in the Data and Cleaning/Formatting

Step # 1: Pull data from the WHO API

In [18]:
import requests

url = 'https://ghoapi.azureedge.net/api/LIFE_0000000035'

try:
    # adding parameters to handle chunked responses more effectively
    response = requests.get(url, stream=True, timeout=10)

    # check if the response is successful
    if response.ok:
        data = response.json()
        print("Data successfully retrieved from WHO API.")
    else:
        print(f"Failed to retrieve data. Status Code: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"An error occured: {e}")
    

An error occured: HTTPSConnectionPool(host='ghoapi.azureedge.net', port=443): Read timed out. (read timeout=10)


Explanation: Retrieved the data from the WHO API, handling potential issues like timeouts and chunked responses.

Step # 2: Convert the JSON response to a DataFrame

In [6]:
import pandas as pd

df = pd.json_normalize(data['value'])
print("Converted the JSON respinse to a DataFrame.")
print(df.head())

NameError: name 'data' is not defined

Explanation: The JSON data from the API is converted into a pandas DataFrame for easy analysis.

Step # 3: Check and remove duplicate rows

In [42]:
df.drop_duplicates(inplace=True)
print(f"Number of rows after removing duplicates: {df.shape[0]}")

Number of rows after removing duplicates: 245784


Explanation: This step ensures no duplicate records are present in the DataFrame. The output shows there were no duplicate rows found.

Step # 4: Handle missing values in the 'NumericValue' column

In [45]:
if df['NumericValue'].isnull().any():
    median_value = df['NumericValue'].median()
    df['NumericValue'].fillna(median_value, inplace=True)
    print("Replaced missing values in 'NumericValue' with median.")
else:
    print("No missing values found in 'NumericValue'.")

No missing values found in 'NumericValue'.


Explanation: If there are any missing values in 'NumericValue' (life expectancy) they are replaced with the median to maintain consistency.

Step # 5: Filter data to include only the years 2003 to 2023

In [48]:
# Step 5: Filter data by years 2003 to 2023
df = df[(df['TimeDim'] >= 2003) & (df['TimeDim'] <= 2023)]
print(f"Number of rows after filtering by years 2003-2023: {df.shape[0]}")
print(df.head())


Number of rows after filtering by years 2003-2023: 212268
        Id    IndicatorCode        SpatialDimType SpatialDim TimeDimType  \
0  2023348  LIFE_0000000035               COUNTRY        ZMB        YEAR   
1  2023354  LIFE_0000000035               COUNTRY        LCA        YEAR   
2  2023417  LIFE_0000000035  WORLDBANKINCOMEGROUP     WB_UMI        YEAR   
3  2023442  LIFE_0000000035               COUNTRY        IRQ        YEAR   
4  2023502  LIFE_0000000035               COUNTRY        STP        YEAR   

  ParentLocationCode         ParentLocation Dim1Type  TimeDim      Dim1  ...  \
0                AFR                 Africa      SEX     2018   SEX_MLE  ...   
1                AMR               Americas      SEX     2018  SEX_BTSX  ...   
2               None                   None      SEX     2013   SEX_MLE  ...   
3                EMR  Eastern Mediterranean      SEX     2013  SEX_BTSX  ...   
4                AFR                 Africa      SEX     2015  SEX_BTSX  ...   

  Da

Explanation: The DataFrame is filtered to include only records from 2003 to 2023.

Step # 6: Rename Columns for Clarity

In [51]:
df.rename(columns={
    'SpatialDim': 'Country_Code',
    'TimeDim': 'Year',
    'NumericValue': 'Life_Expectancy'
}, inplace=True)
print("Renamed columns for better clarity.")
print(df.head())

Renamed columns for better clarity.
        Id    IndicatorCode        SpatialDimType Country_Code TimeDimType  \
0  2023348  LIFE_0000000035               COUNTRY          ZMB        YEAR   
1  2023354  LIFE_0000000035               COUNTRY          LCA        YEAR   
2  2023417  LIFE_0000000035  WORLDBANKINCOMEGROUP       WB_UMI        YEAR   
3  2023442  LIFE_0000000035               COUNTRY          IRQ        YEAR   
4  2023502  LIFE_0000000035               COUNTRY          STP        YEAR   

  ParentLocationCode         ParentLocation Dim1Type  Year      Dim1  ...  \
0                AFR                 Africa      SEX  2018   SEX_MLE  ...   
1                AMR               Americas      SEX  2018  SEX_BTSX  ...   
2               None                   None      SEX  2013   SEX_MLE  ...   
3                EMR  Eastern Mediterranean      SEX  2013  SEX_BTSX  ...   
4                AFR                 Africa      SEX  2015  SEX_BTSX  ...   

  DataSourceDim     Value Life_E

Explanation: This step simplifies column names, making the data easier to understand.

Step # 7: Check for outliers

In [54]:
# Calculate Z-scores to identify outliers
from scipy import stats

df['Z_Score'] = stats.zscore(df['Life_Expectancy'])
outliers = df[df['Z_Score'].abs() > 3]  # Outliers with Z-scores > 3

print("Identified potential outliers in life expectancy.")
print(outliers.head())


Identified potential outliers in life expectancy.
Empty DataFrame
Columns: [Id, IndicatorCode, SpatialDimType, Country_Code, TimeDimType, ParentLocationCode, ParentLocation, Dim1Type, Year, Dim1, Dim2Type, Dim2, Dim3Type, Dim3, DataSourceDimType, DataSourceDim, Value, Life_Expectancy, Low, High, Comments, Date, TimeDimensionValue, TimeDimensionBegin, TimeDimensionEnd, Z_Score]
Index: []

[0 rows x 26 columns]


Explanation: This code calculates the Z-scores for the 'Life_Expectancy' column to identify outliers, where Z-scores measure how many standard deviations a data point is from the mean. Records with an absolute Z-score greater than 3 are considered potential outliers, highlighting unusually high or low life expectancy values compared to the rest of the data.

Step # 8: Save the transformations to a CSV file

In [20]:
print("Preview of final DataFrame to be saved:")
print(df.head())
print(f"Shape of final DataFrame: {df.shape}")


Preview of final DataFrame to be saved:
        Id    IndicatorCode        SpatialDimType Country_Code TimeDimType  \
0  2023348  LIFE_0000000035               COUNTRY          ZMB        YEAR   
1  2023354  LIFE_0000000035               COUNTRY          LCA        YEAR   
2  2023417  LIFE_0000000035  WORLDBANKINCOMEGROUP       WB_UMI        YEAR   
3  2023442  LIFE_0000000035               COUNTRY          IRQ        YEAR   
4  2023502  LIFE_0000000035               COUNTRY          STP        YEAR   

  ParentLocationCode         ParentLocation Dim1Type  Year      Dim1  ...  \
0                AFR                 Africa      SEX  2018   SEX_MLE  ...   
1                AMR               Americas      SEX  2018  SEX_BTSX  ...   
2                NaN                    NaN      SEX  2013   SEX_MLE  ...   
3                EMR  Eastern Mediterranean      SEX  2013  SEX_BTSX  ...   
4                AFR                 Africa      SEX  2015  SEX_BTSX  ...   

      Value Life_Expectancy 

In [5]:
df.to_csv('final_transformed_life_expectancy.csv', index=False)
print("Final transformed dataset saved as 'final_transformed_life_expectancy.csv'.")

NameError: name 'df' is not defined

Step # 9: Reload dataframe from the saved CSV

In [8]:
import pandas as pd

# Load the DataFrame from a CSV file
df = pd.read_csv('final_transformed_life_expectancy.csv')

Step # 9: Display Human_Readable DataFrame

In [11]:
print("Final Transformed Life Expectancy Dataset:")
print(df.head(10))  

Final Transformed Life Expectancy Dataset:
        Id    IndicatorCode        SpatialDimType Country_Code TimeDimType  \
0  2023348  LIFE_0000000035               COUNTRY          ZMB        YEAR   
1  2023354  LIFE_0000000035               COUNTRY          LCA        YEAR   
2  2023417  LIFE_0000000035  WORLDBANKINCOMEGROUP       WB_UMI        YEAR   
3  2023442  LIFE_0000000035               COUNTRY          IRQ        YEAR   
4  2023502  LIFE_0000000035               COUNTRY          STP        YEAR   
5  2023538  LIFE_0000000035               COUNTRY          HND        YEAR   
6  2023580  LIFE_0000000035               COUNTRY          TKM        YEAR   
7  2023589  LIFE_0000000035               COUNTRY          GEO        YEAR   
8  2023638  LIFE_0000000035               COUNTRY          ALB        YEAR   
9  2023642  LIFE_0000000035               COUNTRY          IRQ        YEAR   

  ParentLocationCode         ParentLocation Dim1Type  Year      Dim1  ...  \
0                AFR 

In this project, I worked with life expectancy data from the WHO Global Health Observatory API. I cleaned up the data by focusing on the years 2003-2023, combining information by gender, and filling in any missing values. The data came from the World Health Organization, which collects and shares reliable health information. However, changing the data can sometimes make it less accurate, especially when averaging results across different countries. For example, filling in missing values with an average might not truly represent what's happening in every country. To keep things clear and honest, I documented all the changes I made. While I got the data ethically, it’s important to check it against other sources to ensure it’s accurate and fair.